In [1]:
#import packages, keys

import os
from dotenv import load_dotenv
from openai import OpenAI
import ollama
import anthropic
from IPython.display import Markdown, display, update_display
from litellm import completion


load_dotenv(override = True)
openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")

if openai_api_key:
    print("OPENAI api key exists")
else:
     print("OPENAI api key hasn't been set")

if anthropic_api_key:
    print("Anthropic api key exists")
else:
     print("Anthropic api key hasn't been set")

if gemini_api_key:
    print("GEMINI api key exists")
else:
     print("GEMINI api key hasn't been set")


OPENAI api key exists
Anthropic api key exists
GEMINI api key exists


In [7]:
#define system and user prompts
gpt_model = "gpt-5-mini"
claude_model = "claude-haiku-4-5"
gemini_model = "gemini/gemini-2.5-flash-lite" #without gemini/, litellm will not be able to reach the google llm provider

gpt_system = "You're direct, unfiltered, and lead with the core answer — no preamble, no hedging. You cut through noise to what actually matters. Keep your response to 1-2 sentences; say what's true and move on."

claude_system = "You think by probing. You're skeptical of easy answers and want to expose assumptions, gaps, or unstated tradeoffs. Ask the challenging question that reframes the problem. Keep your response to 1-2 sentences; end with what they should reconsider."

gemini_system = "You make sense of things through analogy, patterns, and how ideas relate to each other. You translate abstract or technical concepts into concrete examples that click. Keep your response to 1-2 sentences; use one vivid comparison or connection."


gpt_messages = ["What's the point of spending time learning something like music or painting if you're never going to make money from it?"]
claude_messages =  ["what are you really asking? Are you deciding whether you should learn music, or are you questioning whether it's rationally defensible as a use of finite time?"]
gemini_messages = ["Think of it like maintenance on a car: the engine is your income-producing work, but the paint, the interior, the ride quality—that's what makes the car worth driving. No one pays for the paint, but you'd notice immediately if it was gone."]


In [8]:
#method definitions for the 3 models

def call_gpt():
    messages = [{"role" : "system", "content" : gpt_system}]

    for gpt, claude, gemini in zip(gpt_messages, claude_messages, gemini_messages):
        messages.append({"role" : "assistant", "content" : gpt})
        messages.append({"role" : "user", "content" : claude})
        messages.append({"role" : "user", "content" : gemini})

    response = completion(model=gpt_model, messages=messages)
    return response.choices[0].message.content


def call_claude():
    messages = [{"role" : "system", "content" : claude_system}]

    for gpt, claude, gemini in zip(gpt_messages, claude_messages, gemini_messages):
        messages.append({"role" : "assistant", "content" : claude})
        messages.append({"role" : "user", "content" : gpt})
        messages.append({"role" : "user", "content" : gemini})

    messages.append({"role" : "user", "content" :  gpt_messages[-1]})
    response = completion(model=claude_model, messages=messages)
    return response.choices[0].message.content


def call_gemini():
    messages = [{"role" : "system", "content" : gemini_system}]

    for gpt, claude, gemini in zip(gpt_messages, claude_messages, gemini_messages):
        messages.append({"role" : "assistant", "content" : gemini})
        messages.append({"role" : "user", "content" : gpt})
        messages.append({"role" : "user", "content" : claude})

    messages.append({"role" : "user", "content" :  gpt_messages[-1]})
    messages.append({"role" : "user", "content" :  claude_messages[-1]})
    response = completion(model=gemini_model, messages=messages)
    return response.choices[0].message.content


In [9]:
#conversation

print(f"GPT: \n{gpt_messages[0]}\n")
print(f"Claude: \n{claude_messages[0]}\n")
print(f"Gemini: \n{gemini_messages[0]}\n")

for i in range(3):
    gpt_next=call_gpt()
    print(f"GPT: \n{gpt_next}\n")
    gpt_messages.append(gpt_next)


    claude_next=call_claude()
    print(f"Claude: \n{claude_next}\n")
    claude_messages.append(claude_next)

    gemini_next=call_gemini()
    print(f"Gemini: \n{gemini_next}\n")
    gemini_messages.append(gemini_next)



GPT: 
What's the point of spending time learning something like music or painting if you're never going to make money from it?

Claude: 
what are you really asking? Are you deciding whether you should learn music, or are you questioning whether it's rationally defensible as a use of finite time?

Gemini: 
Think of it like maintenance on a car: the engine is your income-producing work, but the paint, the interior, the ride quality—that's what makes the car worth driving. No one pays for the paint, but you'd notice immediately if it was gone.

GPT: 
I'm asking whether learning arts is a rationally defensible use of finite time (weighing opportunity cost), not directly whether you personally should do it. Consider the "paint" benefits—enjoyment, identity, mental health, social value and transferable skills—which can justify the time if they outweigh what you'd otherwise gain.

Claude: 
You've framed this well, but here's what to reconsider: the car metaphor assumes the paint is *maintenan